# 06_evaluation

## Setup

In [ ]:
# Mount colab
if True:
    import os
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

# Paths
from pathlib import Path
PROJECT_ROOT: str = f"/content/drive/MyDrive/colab_notebooks/projects/trial_translator_project/v1.2"

In [ ]:
# Install reqs
if True:
    %pip install -r {PROJECT_ROOT + "/requirements.txt"} \
        llama-cpp-python==0.3.20 \
        --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
# -- env libraries --
import os
import sys
import time
import torch
import importlib
from typing import List
from dataclasses import dataclass, asdict, field
from pprint import pprint
from tqdm.auto import tqdm

# -- data science libraries --
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -- hugging face --
from datasets import load_dataset

# -- project specific third-party libraries --
from llama_cpp import Llama
from litellm import completion

# Add the PROJECT_ROOT directory to the Python path to import modules
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# -- project imports --
import src

# Reload all relevant modules
importlib.reload(src)

print("✅All relevant modules have been imported/reloaded.")

In [ ]:
# Config

from dataclasses import dataclass, asdict, field
from typing import Any
import os
import torch

@dataclass
class EnvConfig:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    is_kaggle: bool = "KAGGLE_" in "".join(os.environ.keys())
    smoke_test: bool = False
    requires_keys: bool = True

@dataclass
class DataConfig:
    data_path: str = PROJECT_ROOT + "/data/clinical-trials-bronze-raw.csv"
    gold_dataset_repo_id: str = "Francesco-A/clinical-policy-gold-distill-cot"
    generations_path: str = PROJECT_ROOT + "/artifacts/test_generations.csv"
    eval_path: str = PROJECT_ROOT + "/artifacts/eval.csv"

@dataclass
class ModelConfig:
    system_prompt: str = ""
    judge_prompt: str = ""
    base_model: str = "Qwen/Qwen3-4B-GGUF"
    student_model: str = "Francesco-A/Qwen3-4B-Clinical-Reasoning-Lite-GGUF"
    teacher_1: str = "groq/meta-llama/llama-4-scout-17b-16e-instruct"
    teacher_2: str = "gemini/gemma-3-27b-it"
    judge: str = "groq/openai/gpt-oss-120b"
    context_window: int = 2048
    params: dict[str, Any] = field(default_factory=lambda: {
        "temperature": 0.0,
        "max_tokens": 1024,
    })

@dataclass
# NOTE: Use default_factory for mutable default values to prevent unintended shared state.
class CFG:
    env: EnvConfig = field(default_factory=EnvConfig)
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)

    def __post_init__(self) -> None:
        """Runs automatically after the class is initialized."""
        # 1. Load keys (if necessary)
        if self.env.requires_keys:
            if self.env.is_kaggle:
                try:
                    from kaggle_secrets import UserSecretsClient
                    os.environ["REPO_ID"] = UserSecretsClient().get_secret("REPO_ID")
                    # os.environ["HUGGINGFACE_READ_TOKEN"] = UserSecretsClient().get_secret("HUGGINGFACE_READ_TOKEN")
                    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
                    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
                    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
                    print("✅ All Kaggle Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Kaggle secrets not found or an error occurred: {e}")
            else:
                try:
                    from google.colab import userdata
                    os.environ["REPO_ID"] = userdata.get("REPO_ID")
                    # os.environ["HUGGINGFACE_READ_TOKEN"] = userdata.get("HUGGINGFACE_READ_TOKEN")
                    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
                    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
                    os.environ["HF_TOKEN"] = userdata.get("HF_WRITE_TOKEN")
                    print("✅ All Colab Userdata Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Colab secrets not found or an error occurred: {e}")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


In [ ]:
# Initialize config
config = CFG()
print(f"Using device: {config.env.device}")
pprint(config.to_dict())

In [ ]:
config.model.system_prompt = """
    You are an expert assistant specialized in summarizing clinical trial reports for policy makers.

    Your task is to produce clear, concise, and accurate summaries that highlight:
    - the objective of the study
    - the intervention and population
    - the main findings
    - the policy-relevant implications
    - any important limitations

    Avoid technical jargon when possible. Do not add information that is not present in the input. Do not speculate.

    Write in a neutral, professional tone. The summary should be easy to understand for a non-specialist audience.
    For particularly complex inputs, you may include a brief, structured thought process (e.g., using <think> tags) *before* the final summary to demonstrate your reasoning. Otherwise, provide your summary directly without internal monologues or conversational adornments.
"""

config.model.judge_prompt = """
    You are an expert evaluator of clinical trial summaries written for policy makers.

    You will be given:
    - a clinical trial description
    - two candidate summaries (A and B)

    Your task:
    1. Decide which summary is better for policy makers.
    2. Consider:
    - factual correctness
    - completeness
    - clarity and readability
    - relevance to policy implications
    - conciseness (do not privilege more verbose outputs, a shorter summary that captures all key points is better)

    Rules:
    - Do not assume any information not present in the input.
    - Be strict about hallucinations.
    - Return ONLY valid JSON.
    - Do not engage in conversation.

    Output format:
    {
    "winner": "A" or "B",
    "reason": "short explanation"
    }

    Make sure to ALWAYS start the generation with: {"winner":

"""

## Data

### Load data

In [ ]:
# Read data and display length statistics
trials_df = pd.read_csv(config.data.data_path).fillna("None")

# Create the 'document' column based on the condition and assign the result
trials_df['document'] = trials_df.apply(
    lambda x: "TITLE: " + x['briefTitle'] + "\n\nSUMMARY: " + x['briefSummary']
    if x['brief_len'] >= 100
    else "TITLE: " + x['briefTitle'] + "\n\nSUMMARY: " + x['briefSummary'] + "\n\nDETAILS: " + x['detailedDescription'],
    axis=1
)
trials_df['full_len'] = trials_df[['brief_len','detailed_len']].sum(axis=1)
display(
    trials_df[['brief_len','detailed_len','full_len']]
    .describe(percentiles=[0.25, 0.75, 0.9,])
    )

In [ ]:
# Get training dataset from Hugging Face
clinical_trial_dataset = load_dataset(config.data.gold_dataset_repo_id, split = "train")

# Get ids
training_ids = list(clinical_trial_dataset["nctId"])

### Downsample

In [ ]:
# Select n samples per category within a quantile range (token efficiency)
p25, p90 = trials_df['full_len'].quantile([.25,.90])
downsampled_trials_df = trials_df.query("full_len >= @p25 and full_len <= @p90")
downsampled_trials_df['search_term_used'].value_counts()
print(f"--Retaining only texts with condition: {p90} >= length >= {p25} -- ")

# Sample n samples per each search_term_used
n_sample = 5
print("Downsampling dataset while exluding samples used in training data...")
downsampled_trials_df = (
    downsampled_trials_df[~downsampled_trials_df['nctId'].isin(training_ids)]
    .groupby('search_term_used')
    .apply(lambda x: x.sample(n_sample, random_state=config.env.seed))
    .reset_index(drop=True)
    )

# Check
downsampled_trials_df['search_term_used'].value_counts()

## Generate outputs

### Load models

In [ ]:
# Load the model normally
student_llm = Llama.from_pretrained(
    repo_id=config.model.student_model,
    filename="*Q4_K_M.gguf",
    chat_format="qwen",
    n_gpu_layers=-1,
    n_ctx=config.model.context_window,
    verbose=False,
)

base_llm = Llama.from_pretrained(
    repo_id=config.model.base_model,
    filename="*Q4_K_M.gguf",
    chat_format="qwen",
    n_gpu_layers=-1,
    n_ctx=config.model.context_window,
    verbose=False,
)

def get_local_response(prompt, model, params, system_prompt={'temperature': 0.0, 'max_tokens': 1024}):
    response = model.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
             {"role": "user", "content": prompt}
            ],
        **params
    )

    return response["choices"][0]["message"]["content"]

def get_remote_response(prompt, model, params, system_prompt={'temperature': 0.0, 'max_tokens': 1024}):
    response = completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
            ],
        **params
    )

    return response.choices[0].message.content

### Check actual token length

In [ ]:
# Calculate token lengths for each document
system_prompt_length = len(student_llm.tokenize(config.model.system_prompt.encode('utf-8')))
document_token_lengths = [
    len(student_llm.tokenize(doc.encode('utf-8'))) + system_prompt_length
    for doc in downsampled_trials_df['document'].to_list()
]

# Visualize the distribution of text lengths
plt.figure(figsize=(10, 6))
ax = sns.histplot(
    document_token_lengths,
    bins=5,
    kde=True,
    # log_scale=True
)
plt.show()

### Generate responses & save results

In [ ]:
import re

def strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip().strip("\n")

In [ ]:
if not os.path.exists(config.data.generations_path):
    print("Generating outputs...")

    systems = {
        "base": lambda x: get_local_response(x, base_llm, config.model.params),
        "student": lambda x: get_local_response(x, student_llm, config.model.params),
        "teacher_1": lambda x: get_remote_response(x, config.model.teacher_1, config.model.params),
        "teacher_2": lambda x: get_remote_response(x, config.model.teacher_2, config.model.params),
    }

    results = []

    for sample_dict in tqdm(
        downsampled_trials_df.to_dict('records'),
        desc="Generating outputs...",
        ):
        row = {"id": sample_dict["nctId"], "input": sample_dict["document"]}

        for name, fn in systems.items():
            row[name] = fn(sample_dict["document"])

        results.append(row)

    # Turn into df
    generations_df = pd.DataFrame(results)

    # Strip thinking (only the final result must be evaluated)
    for col in ['base', 'student', 'teacher_1', 'teacher_2']:
        generations_df[col] = generations_df[col].apply(strip_think)

    generations_df.to_csv(config.data.generations_path, index=False)

else:
    print("Loading cached outputs...")
    generations_df = pd.read_csv(config.data.generations_path)

In [ ]:
pprint(
    generations_df.iloc[0].to_dict(),
    width=150,
    sort_dicts=False
    )

## LLM-as-judge evaluation


In [ ]:
def build_judge_prompt(sample, out_a, out_b):
    return f"""
    TRIAL:
    {sample["input"]}

    A:
    {out_a}

    B:
    {out_b}
    """

def judge_pair(sample, out_a, out_b, params={'temperature': 0.0, 'max_tokens': 512}):
    prompt = build_judge_prompt(sample, out_a, out_b)

    # response = client.chat.completions.create( # if using groq client
    response = completion(
        model=config.model.judge,
        messages=[
            {"role": "system", "content": config.model.judge_prompt},
            {"role": "user", "content": prompt}
            ],
        **params
    )

    return json.loads(response.choices[0].message.content)

### Pairwise comparisons

In [ ]:
if not os.path.exists(config.data.eval_path):
    print("Generating evaluations...")

    comparisons = []

    pairs = [
        ("base", "student"),
        ("student", "teacher_1"),
        ("student", "teacher_2"),
        # ("base", "teacher_1"), # sanity check
    ]

    for row in tqdm(
        generations_df.to_dict('records'),
        desc="Evaluating outputs..."
        ):
        for a, b in pairs:
            try:
                eval_results = judge_pair(row, row[a], row[b])
                data_entry = {
                    "id": row["id"],
                    "A": a,
                    "B": b,
                    "winner": eval_results["winner"],
                    "reason": eval_results["reason"],
                }
            except Exception as e:
                print(f"Error judging pair {a} vs {b} for id {row['id']}: {e}")
                data_entry = {
                    "id": row["id"],
                    "A": a,
                    "B": b,
                    "winner": eval_results["winner"],
                    "reason": eval_results["reason"],
                }

            comparisons.append(data_entry)
            print(data_entry)
            time.sleep(20) # avoid rate limiting

    # Save evaluations
    eval_df = pd.DataFrame(comparisons)
    eval_df.to_csv(config.data.eval_path, index=False)

else:
    print("Loading cached evaluations...")
    eval_df = pd.read_csv(config.data.eval_path)

### Aggregate results

In [ ]:
win_rates = eval_df.groupby(["A", "B"]).apply(
    lambda x: (x == "A").mean()
    )

In [ ]:
eval_df.groupby(["A", "B"]).apply(
    lambda x: (x == "A").mean()
    )

## Analysis

WIP...

## Requirements

In [ ]:
!pip freeze

```markdown
absl-py==1.4.0
accelerate==1.12.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.1
anywidget==0.9.21
apsw==3.51.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.2.23.0.48.33
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.8
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.36.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.0.0
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
cffi==2.0.0
chardet==5.2.0
charset-normalizer==3.4.4
clarabel==0.11.1
click==8.3.1
click-plugins==1.1.1.2
cligj==0.7.2
cloudpathlib==0.23.0
cloudpickle==3.1.2
cmake==3.31.10
cmdstanpy==1.3.0
colorcet==3.1.0
colorlover==0.3.0
colour==0.1.5
community==1.0.0b1
confection==0.1.5
cons==0.4.7
contourpy==1.3.3
cramjam==2.11.0
cryptography==43.0.3
cufflinks==0.17.3
curl_cffi==0.14.0
cvxopt==1.3.2
cvxpy==1.6.7
cycler==0.12.1
cyipopt==1.5.0
cymem==2.0.13
Cython==3.0.12
dask==2026.1.2
dataproc-spark-connect==1.0.2
datasets==4.0.0
db-dtypes==1.5.0
dbus-python==1.2.18
debugpy==1.8.15
decorator==4.4.2
defusedxml==0.7.1
deprecation==2.1.0
diffusers==0.36.0
dill==0.3.8
distilabel==1.5.3
distro==1.9.0
dlib==19.24.6
dm-tree==0.1.9
docstring_parser==0.17.0
docutils==0.21.2
dopamine_rl==4.1.2
duckdb==1.3.2
earthengine-api==1.5.24
easydict==1.13
editdistance==0.8.1
eerepr==0.1.2
einops==0.8.2
en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl#sha256=1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85
entrypoints==0.4
esda==2.8.1
et_xmlfile==2.0.0
etils==1.13.0
etuples==0.3.10
Farama-Notifications==0.0.4
fastai==2.8.7
fastapi==0.133.1
fastcore==1.12.18
fastdownload==0.0.7
fastjsonschema==2.21.2
fastlite==0.2.4
fastprogress==1.1.5
fasttransform==0.0.2
fastuuid==0.14.0
ffmpy==1.0.0
filelock==3.24.3
fiona==1.10.1
firebase-admin==6.9.0
Flask==3.1.3
flatbuffers==25.12.19
flax==0.11.2
folium==0.20.0
fonttools==4.61.1
fqdn==1.5.1
frozendict==2.4.7
frozenlist==1.8.0
fsspec==2025.3.0
future==1.0.0
gast==0.7.0
gcsfs==2025.3.0
GDAL==3.8.4
gdown==5.2.1
geemap==0.35.3
geocoder==1.38.1
geographiclib==2.1
geopandas==1.1.2
geopy==2.4.1
giddy==2.3.8
gin-config==0.5.0
gitdb==4.0.12
GitPython==3.1.46
glob2==0.7
google==3.0.0
google-adk==1.26.0
google-ai-generativelanguage==0.6.15
google-api-core==2.30.0
google-api-python-client==2.190.0
google-auth==2.47.0
google-auth-httplib2==0.3.0
google-auth-oauthlib==1.2.4
google-cloud-aiplatform==1.139.0
google-cloud-appengine-logging==1.8.0
google-cloud-audit-log==0.4.0
google-cloud-bigquery==3.40.1
google-cloud-bigquery-connection==1.20.0
google-cloud-bigquery-storage==2.36.2
google-cloud-bigtable==2.35.0
google-cloud-core==2.5.0
google-cloud-dataproc==5.25.0
google-cloud-datastore==2.23.0
google-cloud-discoveryengine==0.13.12
google-cloud-firestore==2.23.0
google-cloud-functions==1.22.0
google-cloud-iam==2.21.0
google-cloud-language==2.19.0
google-cloud-logging==3.13.0
google-cloud-monitoring==2.29.1
google-cloud-pubsub==2.35.0
google-cloud-resource-manager==1.16.0
google-cloud-secret-manager==2.26.0
google-cloud-spanner==3.63.0
google-cloud-speech==2.36.1
google-cloud-storage==3.9.0
google-cloud-trace==1.18.0
google-cloud-translate==3.24.0
google-colab @ file:///colabtools/dist/google_colab-1.0.0.tar.gz
google-crc32c==1.8.0
google-genai==1.65.0
google-generativeai==0.8.6
google-pasta==0.2.0
google-resumable-media==2.8.0
googleapis-common-protos==1.72.0
googledrivedownloader==1.1.0
gradio==5.50.0
gradio_client==1.14.0
grain==0.2.16
graphviz==0.21
greenlet==3.3.2
groovy==0.1.2
groq==1.0.0
grpc-google-iam-v1==0.14.3
grpc-interceptor==0.15.4
grpcio==1.78.0
grpcio-status==1.71.2
grpclib==0.4.9
gspread==6.2.1
gspread-dataframe==4.0.0
gym==0.25.2
gym-notices==0.1.0
gymnasium==1.2.3
h11==0.16.0
h2==4.3.0
h5netcdf==1.8.1
h5py==3.15.1
hdbscan==0.8.41
hf-xet==1.3.1
highspy==1.13.1
holidays==0.91
holoviews==1.22.1
hpack==4.1.0
html5lib==1.1
httpcore==1.0.9
httpimport==1.4.1
httplib2==0.31.2
httptools==0.7.1
httpx==0.28.1
httpx-sse==0.4.3
huggingface_hub==1.4.1
humanize==4.15.0
hyperframe==6.1.0
hyperopt==0.2.7
ibis-framework==9.5.0
idna==3.11
ImageIO==2.37.2
imageio-ffmpeg==0.6.0
imagesize==1.4.1
imbalanced-learn==0.14.1
immutabledict==4.3.1
importlib_metadata==8.7.1
importlib_resources==6.5.2
imutils==0.5.4
inequality==1.1.2
inflect==7.5.0
iniconfig==2.3.0
intel-cmplr-lib-ur==2025.3.2
intel-openmp==2025.3.2
ipyevents==2.0.4
ipyfilechooser==0.6.0
ipykernel==6.17.1
ipyleaflet==0.20.0
ipyparallel==8.8.0
ipython==7.34.0
ipython-genutils==0.2.0
ipython-sql==0.5.0
ipytree==0.2.2
ipywidgets==7.7.1
isoduration==20.11.0
itsdangerous==2.2.0
jaraco.classes==3.4.0
jaraco.context==6.1.0
jaraco.functools==4.4.0
jax==0.7.2
jaxlib==0.7.2
jeepney==0.9.0
jieba==0.42.1
Jinja2==3.1.6
jiter==0.13.0
joblib==1.5.3
jsonpatch==1.33
jsonpickle==4.1.1
jsonpointer==3.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-console==6.6.3
jupyter-events==0.12.0
jupyter-leaflet==0.20.0
jupyter_client==7.4.9
jupyter_core==5.9.1
jupyter_kernel_gateway @ git+https://github.com/googlecolab/kernel_gateway@b134e9945df25c2dcb98ade9129399be10788671
jupyter_server==2.14.0
jupyter_server_terminals==0.5.4
jupyterlab_pygments==0.3.0
jupyterlab_widgets==3.0.16
jupytext==1.19.1
kaggle==1.7.4.5
kagglehub==0.3.13
keras==3.10.0
keras-hub==0.21.1
keras-nlp==0.21.1
keyring==25.7.0
keyrings.google-artifactregistry-auth==1.1.2
kiwisolver==1.4.9
langchain==1.2.10
langchain-core==1.2.16
langgraph==1.0.9
langgraph-checkpoint==4.0.0
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.9
langsmith==0.7.7
lark==1.3.1
launchpadlib==1.10.16
lazr.restfulclient==0.14.4
lazr.uri==1.0.6
lazy_loader==0.4
libclang==18.1.1
libpysal==4.14.1
librosa==0.11.0
lightgbm==4.6.0
linkify-it-py==2.0.3
litellm==1.82.0
llvmlite==0.43.0
locket==1.0.0
logical-unification==0.4.7
lxml==6.0.2
Mako==1.3.10
mapclassify==2.10.0
Markdown==3.10.2
markdown-it-py==4.0.0
MarkupSafe==3.0.3
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
mcp==1.26.0
mdit-py-plugins==0.5.0
mdurl==0.1.2
mgwr==2.2.1
miniKanren==1.0.5
missingno==0.5.2
mistune==3.2.0
mizani==0.13.5
mkl==2025.3.1
ml_dtypes==0.5.4
mlxtend==0.23.4
mmh3==5.2.0
momepy==0.11.0
more-itertools==10.8.0
moviepy==1.0.3
mpmath==1.3.0
msgpack==1.1.2
multidict==6.7.1
multipledispatch==1.0.0
multiprocess==0.70.16
multitasking==0.0.12
murmurhash==1.0.15
music21==9.9.1
namex==0.1.0
narwhals==2.17.0
natsort==8.4.0
nbclassic==1.3.3
nbclient==0.10.4
nbconvert==7.17.0
nbformat==5.10.4
ndindex==1.10.1
nest-asyncio==1.6.0
networkx==3.6.1
nibabel==5.3.3
nltk==3.9.1
notebook==6.5.7
notebook_shim==0.2.4
numba==0.60.0
numexpr==2.14.1
numpy==2.0.2
nvidia-nccl-cu12==2.29.3
oauth2client==4.1.3
oauthlib==3.3.1
omegaconf==2.3.0
onemkl-license==2025.3.1
openai==2.24.0
opencv-contrib-python==4.13.0.92
opencv-python==4.13.0.92
opencv-python-headless==4.13.0.92
openpyxl==3.1.5
opentelemetry-api==1.38.0
opentelemetry-exporter-gcp-logging==1.11.0a0
opentelemetry-exporter-gcp-monitoring==1.11.0a0
opentelemetry-exporter-gcp-trace==1.11.0
opentelemetry-exporter-otlp-proto-common==1.38.0
opentelemetry-exporter-otlp-proto-http==1.38.0
opentelemetry-proto==1.38.0
opentelemetry-resourcedetector-gcp==1.11.0a0
opentelemetry-sdk==1.38.0
opentelemetry-semantic-conventions==0.59b0
opt_einsum==3.4.0
optax==0.2.7
optree==0.19.0
orbax-checkpoint==0.11.33
orjson==3.11.7
ormsgpack==1.12.2
osqp==1.1.1
overrides==7.7.0
packaging==26.0
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
pandocfilters==1.5.1
panel==1.8.7
param==2.3.2
parso==0.8.6
parsy==2.2
partd==1.4.2
pathlib_abc==0.5.2
patsy==1.0.2
peewee==4.0.0
peft==0.18.1
pexpect==4.9.0
pickleshare==0.7.5
pillow==11.3.0
platformdirs==4.9.2
plotly==5.24.1
plotnine==0.14.5
pluggy==1.6.0
plum-dispatch==2.7.1
pointpats==2.5.5
polars==1.35.2
polars-runtime-32==1.35.2
pooch==1.9.0
portalocker==3.2.0
portpicker==1.5.2
preshed==3.0.12
prettytable==3.17.0
proglog==0.1.12
progressbar2==4.5.0
prometheus_client==0.24.1
promise==2.3
prompt_toolkit==3.0.52
propcache==0.4.1
prophet==1.3.0
proto-plus==1.27.1
protobuf==5.29.6
psutil==5.9.5
psycopg2==2.9.11
psygnal==0.15.1
ptyprocess==0.7.0
PuLP==3.3.0
py-cpuinfo==9.0.0
py4j==0.10.9.9
pyarrow==18.1.0
pyasn1==0.6.2
pyasn1_modules==0.4.2
pycairo==1.29.0
pycocotools==2.0.11
pycparser==3.0
pycryptodomex==3.23.0
pydantic==2.12.3
pydantic-settings==2.13.1
pydantic_core==2.41.4
pydata-google-auth==1.9.1
pydot==4.0.1
pydotplus==2.0.2
PyDrive2==1.21.3
pydub==0.25.1
pyerfa==2.0.1.5
pygame==2.6.1
pygit2==1.19.1
Pygments==2.19.2
PyGObject==3.48.2
pyiceberg==0.11.0
PyJWT==2.11.0
pymc==5.28.0
pynndescent==0.6.0
pyogrio==0.12.1
pyomo==6.10.0
PyOpenGL==3.1.10
pyOpenSSL==24.2.1
pyparsing==3.3.2
pyperclip==1.11.0
pyproj==3.7.2
pyroaring==1.0.3
pysal==25.7
pyshp==3.0.3
PySocks==1.7.1
pyspark==4.0.2
pytensor==2.38.0
pytest==8.4.2
python-apt==0.0.0
python-box==7.4.1
python-dateutil==2.9.0.post0
python-dotenv==1.2.1
python-fasthtml==0.12.47
python-json-logger==4.0.0
python-louvain==0.16
python-multipart==0.0.22
python-slugify==8.0.4
python-snappy==0.7.3
python-utils==3.9.1
pytz==2025.2
pyviz_comms==3.0.6
PyWavelets==1.9.0
PyYAML==6.0.3
pyzmq==26.2.1
quantecon==0.11.0
rasterio==1.5.0
rasterstats==0.20.0
ratelim==0.1.6
referencing==0.37.0
regex==2025.11.3
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
requirements-parser==0.9.0
rfc3339-validator==0.1.4
rfc3986-validator==0.1.1
rfc3987-syntax==1.1.0
rich==13.9.4
roman-numerals==4.1.0
roman-numerals-py==4.1.0
rpds-py==0.30.0
rpy2==3.5.17
rsa==4.9.1
rtree==1.4.1
ruff==0.15.4
safehttpx==0.1.7
safetensors==0.7.0
scikit-image==0.25.2
scikit-learn==1.6.1
scipy==1.16.3
scooby==0.11.0
scs==3.2.11
seaborn==0.13.2
SecretStorage==3.5.0
segregation==2.5.3
semantic-version==2.10.0
Send2Trash==2.1.0
sentence-transformers==5.2.3
sentencepiece==0.2.1
sentry-sdk==2.53.0
setuptools==75.2.0
shap==0.50.0
shapely==2.1.2
shellingham==1.5.4
simple-parsing==0.1.8
simplejson==3.20.2
simsimd==6.5.13
six==1.17.0
sklearn-compat==0.1.5
sklearn-pandas==2.2.0
slicer==0.0.8
smart_open==7.5.1
smmap==5.0.2
sniffio==1.3.1
snowballstemmer==3.0.1
soundfile==0.13.1
soupsieve==2.8.3
soxr==1.0.0
spacy==3.8.11
spacy-legacy==3.0.12
spacy-loggers==1.0.5
spaghetti==1.7.6
spanner-graph-notebook==1.1.8
spglm==1.1.0
Sphinx==8.2.3
sphinxcontrib-applehelp==2.0.0
sphinxcontrib-devhelp==2.0.0
sphinxcontrib-htmlhelp==2.1.0
sphinxcontrib-jsmath==1.0.1
sphinxcontrib-qthelp==2.0.0
sphinxcontrib-serializinghtml==2.0.0
spint==1.0.7
splot==1.1.7
spopt==0.7.0
spreg==1.8.5
SQLAlchemy==2.0.47
sqlalchemy-spanner==1.17.2
sqlglot==25.20.2
sqlparse==0.5.5
srsly==2.5.2
sse-starlette==3.2.0
stanio==0.5.1
starlette==0.52.1
statsmodels==0.14.6
strictyaml==1.7.3
stringzilla==4.6.0
stumpy==1.13.0
sympy==1.14.0
tables==3.10.2
tabulate==0.9.0
tbb==2022.3.1
tblib==3.2.2
tcmlib==1.4.1
tenacity==9.1.4
tensorboard==2.19.0
tensorboard-data-server==0.7.2
tensorflow==2.19.0
tensorflow-datasets==4.9.9
tensorflow-hub==0.16.1
tensorflow-metadata==1.17.3
tensorflow-probability==0.25.0
tensorflow-text==2.19.0
tensorflow_decision_forests==1.12.0
tensorstore==0.1.81
termcolor==3.3.0
terminado==0.18.1
text-unidecode==1.3
textblob==0.19.0
tf-slim==1.1.0
tf_keras==2.19.0
thinc==8.3.10
threadpoolctl==3.6.0
tifffile==2026.2.24
tiktoken==0.12.0
timm==1.0.25
tinycss2==1.4.0
tobler==0.13.0
tokenizers==0.22.2
toml==0.10.2
tomlkit==0.13.3
toolz==0.12.1
torch @ https://download.pytorch.org/whl/cpu/torch-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchao==0.10.0
torchaudio @ https://download.pytorch.org/whl/cpu/torchaudio-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchcodec @ https://download.pytorch.org/whl/cpu/torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision @ https://download.pytorch.org/whl/cpu/torchvision-0.25.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
tornado==6.5.1
tqdm==4.67.3
traitlets==5.7.1
traittypes==0.2.3
transformers==5.0.0
treescope==0.1.10
tsfresh==0.21.1
tweepy==4.16.0
typeguard==4.5.1
typer==0.24.1
typer-slim==0.24.0
types-pytz==2025.2.0.20251108
types-setuptools==80.9.0.20251223
typing-inspection==0.4.2
typing_extensions==4.15.0
tzdata==2025.3
tzlocal==5.3.1
uc-micro-py==1.0.3
umap-learn==0.5.11
umf==1.0.3
universal_pathlib==0.3.10
uri-template==1.3.0
uritemplate==4.2.0
urllib3==2.5.0
uuid_utils==0.14.1
uvicorn==0.41.0
uvloop==0.22.1
vega-datasets==0.9.0
wadllib==1.3.6
wandb==0.25.0
wasabi==1.1.3
watchdog==6.0.0
watchfiles==1.1.1
wcwidth==0.6.0
weasel==0.4.3
webcolors==25.10.0
webencodings==0.5.1
websocket-client==1.9.0
websockets==15.0.1
Werkzeug==3.1.6
wheel==0.46.3
widgetsnbextension==3.6.10
wordcloud==1.9.6
wrapt==2.1.1
wurlitzer==3.1.1
xarray==2025.12.0
xarray-einstats==0.10.0
xgboost==3.2.0
xlrd==2.0.2
xxhash==3.6.0
xyzservices==2025.11.0
yarl==1.22.0
ydf==0.15.0
yellowbrick==1.5
yfinance==0.2.66
zipp==3.23.0
zstandard==0.25.0
```